In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings

from langchain_chroma import Chroma

from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers import ContextualCompressionRetriever

from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

from langchain_core.prompts import ChatPromptTemplate

In [4]:
dummy_text = """
The company was founded in 2010 by John Doe.

It started as a small shoe store.

Over the years,
it expanded into apparel and electronics.

In 2023,
the CEO announced a new sustainability initiative.

The Q3 profit in 2023 was a record-breaking
$5.2 Million,
driven mainly by electronics sales.

Our headquarters is located in New York.

Branches exist in London and Tokyo.
"""

with open("dummy_data.txt", "w", encoding="utf-8") as f:
    f.write(dummy_text)

In [5]:
loader = TextLoader("dummy_data.txt")

docs = loader.load()

print("Documents:", len(docs))

Documents: 1


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0
)

splits = text_splitter.split_documents(docs)

print("Chunks:", len(splits))

Chunks: 2


In [7]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [8]:
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_directory="chroma_compression_db"
)

In [9]:
base_retriever = vectorstore.as_retriever(
    search_kwargs={
        "k": 2
    }
)

In [10]:
compress_llm = ChatOllama(
    model="mistral",
    temperature=0
)

In [11]:
compressor = LLMChainExtractor.from_llm(
    compress_llm
)

In [12]:
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [13]:
query = "What was the Q3 profit in 2023?"

print("Question:")
print(query)

Question:
What was the Q3 profit in 2023?


In [14]:
print("=" * 80)
print("WITHOUT COMPRESSION")
print("=" * 80)

docs = base_retriever.invoke(query)

for i, doc in enumerate(docs, 1):
    print(f"\nChunk {i}\n")
    print(doc.page_content)

WITHOUT COMPRESSION

Chunk 1

The company was founded in 2010 by John Doe.

It started as a small shoe store.

Over the years,
it expanded into apparel and electronics.

In 2023,
the CEO announced a new sustainability initiative.

The Q3 profit in 2023 was a record-breaking
$5.2 Million,
driven mainly by electronics sales.

Chunk 2

Our headquarters is located in New York.

Branches exist in London and Tokyo.


In [15]:
print("=" * 80)
print("WITH CONTEXTUAL COMPRESSION")
print("=" * 80)

compressed_docs = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_docs, 1):
    print(f"\nCompressed Chunk {i}\n")
    print(doc.page_content)

WITH CONTEXTUAL COMPRESSION

Compressed Chunk 1

The Q3 profit in 2023 was a record-breaking $5.2 Million, driven mainly by electronics sales.

Compressed Chunk 2

NO_OUTPUT (There is no information about Q3 profit in 2023 in the provided context.)


In [16]:
llm = ChatOllama(
    model="mistral",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
Answer the question using only the provided context.

Context:
{context}

Question:
{input}
""")

document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

retrieval_chain = create_retrieval_chain(
    compression_retriever,
    document_chain
)

In [17]:
response = retrieval_chain.invoke(
    {
        "input": "What was the Q3 profit in 2023?"
    }
)

print(response["answer"])

 The Q3 profit in 2023 was $5.2 Million, as stated in the provided context.
